# Aerial Threat Detection — Robustness Evaluation
**Bar Ilan University — Digital Image Processing Course Project**

Evaluating the robustness of image processing and vision algorithms on **aerial aircraft detection** imagery
(military surveillance / threat detection context).

| Choice | Selection |
|---|---|
| Dataset | `keremberke/aerial-airport-object-detection` (aerial images of aircraft) |
| Task 1 (low-level) | ORB keypoint detection — metric: matching ratio |
| Task 2 (high-level DL) | YOLOv8n object detection — metric: detection recall |
| Task 3 (low-level) | Canny edge detection — metric: edge density ratio |
| Distortion 1 | Gaussian Noise |
| Distortion 2 | Low Light |
| Distortion 3 | Rain |
| Enhancement 1 | NLM Denoising + Bilateral Filter |
| Enhancement 2 | Gamma Correction + CLAHE |
| Enhancement 3 | De-raining (Median Blur + Bilateral) |

## 0. Setup & Imports

In [ ]:
# Install dependencies (run once)
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "ultralytics", "albumentations", "datasets",
                "opencv-python-headless", "matplotlib", "numpy", "Pillow"], check=True)
print("Dependencies ready.")

In [ ]:
import random
import numpy as np
import cv2
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from pathlib import Path
from PIL import Image
import torch
import albumentations as A
from datasets import load_dataset
from ultralytics import YOLO
import warnings
warnings.filterwarnings("ignore")

random.seed(7)
np.random.seed(7)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

## 1. Load Dataset

**Dataset**: `keremberke/aerial-airport-object-detection`  
Aerial/satellite images of airports with bounding-box annotations for aircraft.  
Context: military-style aerial surveillance and threat detection.

In [ ]:
random.seed(7)

ds = load_dataset("keremberke/aerial-airport-object-detection", "full", split="test")
N = len(ds)
print(f"Dataset size: {N} images")
print(f"Features: {list(ds.features.keys())}")

# Sample 20 images for our experiments
NUM_SAMPLES = 20
idxs = random.sample(range(N), NUM_SAMPLES)
samples = [ds[i] for i in idxs]
images = [s["image"] for s in samples]  # list of PIL Images

# Extract ground-truth bounding boxes (COCO format: [x, y, w, h])
def get_gt_boxes_xyxy(sample):
    """Convert COCO [x,y,w,h] bboxes to [x1,y1,x2,y2]."""
    objs = sample.get("objects", {})
    if not objs:
        return np.zeros((0, 4))
    bboxes = objs.get("bbox", [])
    result = []
    for b in bboxes:
        x, y, w, h = b
        result.append([x, y, x + w, y + h])
    return np.array(result) if result else np.zeros((0, 4))

gt_boxes = [get_gt_boxes_xyxy(s) for s in samples]
print(f"Loaded {NUM_SAMPLES} samples. Example GT boxes shape: {gt_boxes[0].shape}")

In [ ]:
# Visualize 4 sample images with GT bounding boxes
fig, axes = plt.subplots(1, 4, figsize=(20, 5))
fig.suptitle("Sample Aerial Images with GT Annotations", fontsize=14, fontweight="bold")

for ax, img, boxes in zip(axes, images[:4], gt_boxes[:4]):
    img_np = np.array(img.convert("RGB"))
    ax.imshow(img_np)
    for box in boxes:
        x1, y1, x2, y2 = box
        rect = patches.Rectangle((x1, y1), x2-x1, y2-y1,
                                  linewidth=2, edgecolor='lime', facecolor='none')
        ax.add_patch(rect)
    ax.set_title(f"GT boxes: {len(boxes)}")
    ax.axis("off")
plt.tight_layout()
plt.show()

## Helper Functions

In [ ]:
# ── IoU and detection recall ─────────────────────────────────────────────────

def compute_iou(box1, box2):
    """Compute IoU between two boxes in xyxy format."""
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])
    inter = max(0, x2 - x1) * max(0, y2 - y1)
    area1 = (box1[2] - box1[0]) * (box1[3] - box1[1])
    area2 = (box2[2] - box2[0]) * (box2[3] - box2[1])
    union = area1 + area2 - inter
    return inter / union if union > 0 else 0.0


def detection_recall(ref_boxes, pred_boxes, iou_thresh=0.5):
    """Fraction of GT boxes matched by at least one prediction."""
    if len(ref_boxes) == 0:
        return 1.0  # nothing to detect
    if len(pred_boxes) == 0:
        return 0.0
    matched = 0
    for gt in ref_boxes:
        for pred in pred_boxes:
            if compute_iou(gt, pred) >= iou_thresh:
                matched += 1
                break
    return matched / len(ref_boxes)


# ── SNR ──────────────────────────────────────────────────────────────────────

def compute_snr(clean_rgb, dist_rgb):
    """SNR (dB) = 10 * log10(signal_power / noise_power), noise = clean − distorted."""
    clean = clean_rgb.astype(np.float64)
    noise = clean - dist_rgb.astype(np.float64)
    signal_power = np.mean(clean ** 2)
    noise_power = np.mean(noise ** 2)
    return 10.0 * np.log10(signal_power / noise_power) if noise_power > 0 else np.inf


# ── Distortion helper ────────────────────────────────────────────────────────

def apply_distortion(img_pil, aug):
    img = np.array(img_pil.convert("RGB"))
    return aug(image=img)["image"]


print("Helper functions loaded.")

---
# Part 1 — Baseline on Clean Images

### 1a. Task 1 — ORB Keypoint Detection

In [ ]:
def orb_overlay(img_pil, nfeatures=800):
    img = np.array(img_pil.convert("RGB"))
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    orb = cv2.ORB_create(nfeatures=nfeatures)
    kps = orb.detect(gray, None)
    out = cv2.drawKeypoints(img, kps, None,
                            flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS)
    return out, kps


# Run on first 4 images
fig, axes = plt.subplots(1, 4, figsize=(20, 5))
fig.suptitle("Part 1 — ORB Keypoints on Clean Aerial Images", fontsize=14, fontweight="bold")
clean_kps_counts = []

for i, (ax, img) in enumerate(zip(axes, images[:4])):
    out, kps = orb_overlay(img)
    clean_kps_counts.append(len(kps))
    ax.imshow(out)
    ax.set_title(f"ORB keypoints: {len(kps)}")
    ax.axis("off")
plt.tight_layout()
plt.show()

# Compute for all 20 samples
clean_kps_counts = []
for img in images:
    _, kps = orb_overlay(img)
    clean_kps_counts.append(len(kps))
print(f"Mean ORB keypoints on clean images: {np.mean(clean_kps_counts):.1f}")

### 1b. Task 2 — YOLO Object Detection (Pretrained YOLOv8n)

In [ ]:
yolo_model = YOLO("yolov8n.pt")


def yolo_predict(img_pil, conf=0.25):
    img = np.array(img_pil.convert("RGB"))
    r = yolo_model.predict(img, conf=conf, verbose=False)[0]
    boxes_xyxy = r.boxes.xyxy.cpu().numpy() if r.boxes is not None else np.zeros((0, 4))
    return r.plot(), boxes_xyxy


# Visualise 4 images
fig, axes = plt.subplots(1, 4, figsize=(20, 5))
fig.suptitle("Part 1 — YOLO Detections on Clean Aerial Images", fontsize=14, fontweight="bold")

for ax, img, gt in zip(axes, images[:4], gt_boxes[:4]):
    out, pred = yolo_predict(img)
    rec = detection_recall(gt, pred)
    ax.imshow(out)
    ax.set_title(f"YOLO: {len(pred)} det | recall: {rec:.2f}")
    ax.axis("off")
plt.tight_layout()
plt.show()

# Compute recall for all samples
clean_recall_vals = []
clean_pred_boxes = []
for img, gt in zip(images, gt_boxes):
    _, pred = yolo_predict(img)
    clean_pred_boxes.append(pred)
    clean_recall_vals.append(detection_recall(gt, pred))

clean_recall_mean = float(np.mean(clean_recall_vals))
print(f"Mean detection recall on clean images: {clean_recall_mean:.3f}")

### 1c. Task 3 — Canny Edge Detection

In [ ]:
def canny_edges(img_pil, low=100, high=200):
    gray = cv2.cvtColor(np.array(img_pil.convert("RGB")), cv2.COLOR_RGB2GRAY)
    return cv2.Canny(gray, low, high)


def edge_density(img_pil):
    edges = canny_edges(img_pil)
    return float(edges.sum()) / edges.size


# Visualise 4 images
fig, axes = plt.subplots(1, 4, figsize=(20, 5))
fig.suptitle("Part 1 — Canny Edges on Clean Aerial Images", fontsize=14, fontweight="bold")

for ax, img in zip(axes, images[:4]):
    edges = canny_edges(img)
    ax.imshow(edges, cmap="gray")
    ax.set_title(f"Edge density: {edges.mean():.4f}")
    ax.axis("off")
plt.tight_layout()
plt.show()

# Compute baseline densities
clean_edge_densities = [edge_density(img) for img in images]
clean_edge_mean = float(np.mean(clean_edge_densities))
print(f"Mean edge density on clean images: {clean_edge_mean:.5f}")

### 1d. Baseline Summary

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle("Part 1 — Baseline Performance on Clean Images", fontsize=14, fontweight="bold")

axes[0].bar(["ORB mean keypoints"], [np.mean(clean_kps_counts)], color="steelblue")
axes[0].set_title("Task 1: ORB Keypoints")
axes[0].set_ylabel("Keypoint count")

axes[1].bar(["YOLO mean recall"], [clean_recall_mean], color="darkorange")
axes[1].set_ylim(0, 1.1)
axes[1].set_title("Task 2: YOLO Detection Recall")
axes[1].set_ylabel("Recall")

axes[2].bar(["Canny mean density"], [clean_edge_mean], color="seagreen")
axes[2].set_title("Task 3: Canny Edge Density")
axes[2].set_ylabel("Edge density")

plt.tight_layout()
plt.show()

print(f"\nBaseline summary:")
print(f"  ORB keypoints (mean): {np.mean(clean_kps_counts):.1f}")
print(f"  YOLO recall   (mean): {clean_recall_mean:.3f}")
print(f"  Edge density  (mean): {clean_edge_mean:.5f}")

---
# Part 2 — Performance on Distorted Images

### 2a. Define Distortions

In [ ]:
distortions = {
    "GaussNoise": A.GaussNoise(var_limit=(500.0, 1500.0), p=1.0),
    "LowLight":   A.RandomBrightnessContrast(
                      brightness_limit=(-0.8, -0.6),
                      contrast_limit=(0.0, 0.0), p=1.0),
    "Rain":       A.RandomRain(slant_lower=-10, slant_upper=10, drop_length=20,
                      drop_width=1, drop_color=(200,200,200),
                      blur_value=3, brightness_coefficient=0.9, p=1.0),
}

# Visualise 4 images × 3 distortions
fig, axes = plt.subplots(4, len(distortions), figsize=(14, 14))
fig.suptitle("Part 2 — Distorted Aerial Images", fontsize=14, fontweight="bold")

for row, img in enumerate(images[:4]):
    for col, (name, aug) in enumerate(distortions.items()):
        dist_img = apply_distortion(img, aug)
        axes[row, col].imshow(dist_img)
        axes[row, col].axis("off")
        if row == 0:
            axes[row, col].set_title(name, fontsize=12)
plt.tight_layout()
plt.show()

### 2b. Measure All Tasks on Each Distortion

In [ ]:
dist_results = {}  # dist_name → {"orb": [...], "recall": [...], "edge": [...]}

for dist_name, aug in distortions.items():
    print(f"Evaluating distortion: {dist_name}")
    orb_vals, recall_vals, edge_vals = [], [], []

    for img, gt in zip(images, gt_boxes):
        dist_np = apply_distortion(img, aug)
        dist_pil = Image.fromarray(dist_np)

        # Task 1: ORB
        _, kps = orb_overlay(dist_pil)
        orb_vals.append(len(kps))

        # Task 2: YOLO recall
        _, pred = yolo_predict(dist_pil)
        recall_vals.append(detection_recall(gt, pred))

        # Task 3: edge density
        edge_vals.append(edge_density(dist_pil))

    dist_results[dist_name] = {
        "orb":    np.array(orb_vals),
        "recall": np.array(recall_vals),
        "edge":   np.array(edge_vals),
    }
    print(f"  ORB: {np.mean(orb_vals):.1f}  |  Recall: {np.mean(recall_vals):.3f}  |  Edge: {np.mean(edge_vals):.5f}")

print("\nDone.")

In [ ]:
# ── Bar charts: clean vs each distortion, per task ───────────────────────────
dist_names = list(distortions.keys())
x = np.arange(len(dist_names) + 1)
labels = ["Clean"] + dist_names
colors = ["steelblue", "tomato", "darkorange", "orchid"]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("Part 2 — Performance Degradation Under Distortion", fontsize=14, fontweight="bold")

# Task 1: ORB
orb_vals = [np.mean(clean_kps_counts)] + [np.mean(dist_results[d]["orb"]) for d in dist_names]
axes[0].bar(labels, orb_vals, color=colors)
axes[0].axhline(np.mean(clean_kps_counts), color="red", linestyle="--", label="Clean baseline")
axes[0].set_title("Task 1: ORB Keypoints")
axes[0].set_ylabel("Mean keypoint count")
axes[0].legend()

# Task 2: YOLO recall
recall_vals_bar = [clean_recall_mean] + [np.mean(dist_results[d]["recall"]) for d in dist_names]
for ax_bar, val, label in zip([axes[1]] * len(labels), recall_vals_bar, labels):
    pass  # filled below
axes[1].bar(labels, recall_vals_bar, color=colors)
axes[1].axhline(clean_recall_mean, color="red", linestyle="--", label="Clean baseline")
axes[1].set_ylim(0, 1.1)
axes[1].set_title("Task 2: YOLO Detection Recall")
axes[1].set_ylabel("Mean recall")
axes[1].legend()

# Task 3: Edge density
edge_vals_bar = [clean_edge_mean] + [np.mean(dist_results[d]["edge"]) for d in dist_names]
axes[2].bar(labels, edge_vals_bar, color=colors)
axes[2].axhline(clean_edge_mean, color="red", linestyle="--", label="Clean baseline")
axes[2].set_title("Task 3: Canny Edge Density")
axes[2].set_ylabel("Mean edge density")
axes[2].legend()

plt.tight_layout()
plt.show()

### 2c. SNR Sweep — Performance vs. Distortion Level (LowLight)

In [ ]:
brightness_levels = [-0.1, -0.2, -0.3, -0.4, -0.5, -0.6, -0.7, -0.8, -0.9]
snr_vals, recall_snr_vals, orb_snr_vals, edge_snr_vals = [], [], [], []

for b in brightness_levels:
    aug = A.RandomBrightnessContrast(brightness_limit=(b, b), contrast_limit=(0, 0), p=1.0)
    lvl_snr, lvl_recall, lvl_orb, lvl_edge = [], [], [], []

    for img, gt in zip(images, gt_boxes):
        clean_np = np.array(img.convert("RGB"))
        dist_np = apply_distortion(img, aug)
        dist_pil = Image.fromarray(dist_np)

        lvl_snr.append(compute_snr(clean_np, dist_np))

        _, pred = yolo_predict(dist_pil)
        lvl_recall.append(detection_recall(gt, pred))

        _, kps = orb_overlay(dist_pil)
        lvl_orb.append(len(kps) / max(1, np.mean(clean_kps_counts)))  # ratio

        lvl_edge.append(edge_density(dist_pil) / max(1e-9, clean_edge_mean))  # ratio

    snr_vals.append(float(np.mean(lvl_snr)))
    recall_snr_vals.append(float(np.mean(lvl_recall)))
    orb_snr_vals.append(float(np.mean(lvl_orb)))
    edge_snr_vals.append(float(np.mean(lvl_edge)))

print("SNR sweep done.")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("Part 2 — Performance vs SNR (LowLight Sweep)", fontsize=14, fontweight="bold")

for ax, vals, title, ylabel in zip(
    axes,
    [recall_snr_vals, orb_snr_vals, edge_snr_vals],
    ["YOLO Detection Recall vs SNR", "ORB Matching Ratio vs SNR", "Edge Density Ratio vs SNR"],
    ["Mean recall vs clean", "ORB ratio vs clean", "Edge ratio vs clean"]
):
    ax.plot(snr_vals, vals, marker="o", color="steelblue")
    ax.axhline(1.0, color="red", linestyle="--", label="Clean baseline")
    ax.set_xlabel("SNR (dB)  ← darker")
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.legend()
    ax.grid(True, alpha=0.3)
    # annotate brightness levels
    for snr, b, val in zip(snr_vals, brightness_levels, vals):
        ax.annotate(f"b={b}", (snr, val), textcoords="offset points",
                    xytext=(5, 5), fontsize=7)

plt.tight_layout()
plt.show()

---
# Part 3 — Performance on Restored (Enhanced) Images

### 3a. Enhancement Functions

In [ ]:
def restore_noise(img_rgb):
    """NLM denoising (h=25 for heavy noise) + bilateral smoothing."""
    bgr = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2BGR)
    den = cv2.fastNlMeansDenoisingColored(bgr, None, 25, 25, 7, 35)
    den = cv2.bilateralFilter(den, d=9, sigmaColor=80, sigmaSpace=80)
    return cv2.cvtColor(den, cv2.COLOR_BGR2RGB)


def restore_lowlight(img_rgb):
    """Gamma correction (γ=0.35) + strong CLAHE on L-channel."""
    gamma = 0.35
    lut = (np.arange(256) / 255.0) ** gamma * 255
    lut = np.clip(lut, 0, 255).astype(np.uint8)
    img_gamma = cv2.LUT(img_rgb, lut)
    lab = cv2.cvtColor(img_gamma, cv2.COLOR_RGB2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=6.0, tileGridSize=(8, 8))
    l = clahe.apply(l)
    return cv2.cvtColor(cv2.merge([l, a, b]), cv2.COLOR_LAB2RGB)


def restore_rain(img_rgb):
    """De-raining: median blur removes streak artefacts, bilateral restores edges."""
    derained = cv2.medianBlur(img_rgb, 3)
    derained = cv2.bilateralFilter(derained, d=9, sigmaColor=75, sigmaSpace=75)
    return derained


restorers = {
    "GaussNoise": restore_noise,
    "LowLight":   restore_lowlight,
    "Rain":       restore_rain,
}

print("Enhancement functions loaded.")

### 3b. Visual Comparison: Clean / Distorted / Restored

In [ ]:
for dist_name, aug in distortions.items():
    restore_fn = restorers[dist_name]
    fig, axes = plt.subplots(2, 3, figsize=(15, 8))
    fig.suptitle(f"Distortion: {dist_name}  |  Clean / Distorted / Restored",
                 fontsize=13, fontweight="bold")

    for row, img in enumerate(images[:2]):
        dist_np = apply_distortion(img, aug)
        rest_np = restore_fn(dist_np)

        axes[row, 0].imshow(np.array(img.convert("RGB")))
        axes[row, 0].set_title("Clean"); axes[row, 0].axis("off")

        axes[row, 1].imshow(dist_np)
        axes[row, 1].set_title("Distorted"); axes[row, 1].axis("off")

        axes[row, 2].imshow(rest_np)
        axes[row, 2].set_title("Restored"); axes[row, 2].axis("off")

    plt.tight_layout()
    plt.show()

### 3c. Measure Tasks on Enhanced Images

In [ ]:
enh_results = {}  # dist_name → {"orb": [...], "recall": [...], "edge": [...]}

for dist_name, aug in distortions.items():
    restore_fn = restorers[dist_name]
    print(f"Evaluating enhancement for: {dist_name}")
    orb_vals, recall_vals, edge_vals = [], [], []

    for img, gt in zip(images, gt_boxes):
        dist_np = apply_distortion(img, aug)
        rest_np = restore_fn(dist_np)
        rest_pil = Image.fromarray(rest_np)

        _, kps = orb_overlay(rest_pil)
        orb_vals.append(len(kps))

        _, pred = yolo_predict(rest_pil)
        recall_vals.append(detection_recall(gt, pred))

        edge_vals.append(edge_density(rest_pil))

    enh_results[dist_name] = {
        "orb":    np.array(orb_vals),
        "recall": np.array(recall_vals),
        "edge":   np.array(edge_vals),
    }
    print(f"  ORB: {np.mean(orb_vals):.1f}  |  Recall: {np.mean(recall_vals):.3f}  |  Edge: {np.mean(edge_vals):.5f}")

print("\nDone.")

In [ ]:
# Comparison bar charts: Distorted vs Enhanced, per distortion and task
fig, axes = plt.subplots(3, 3, figsize=(18, 13))
fig.suptitle("Part 3 — Distorted vs Enhanced Performance", fontsize=14, fontweight="bold")

task_keys  = ["orb",     "recall",  "edge"]
task_names = ["Task 1: ORB keypoints", "Task 2: YOLO recall", "Task 3: Edge density"]
clean_baselines = [np.mean(clean_kps_counts), clean_recall_mean, clean_edge_mean]

for col, (task_key, task_name, clean_val) in enumerate(
        zip(task_keys, task_names, clean_baselines)):
    for row, dist_name in enumerate(dist_names):
        ax = axes[row, col]
        dist_val = float(np.mean(dist_results[dist_name][task_key]))
        enh_val  = float(np.mean(enh_results[dist_name][task_key]))

        ax.bar(["Distorted", "Enhanced"], [dist_val, enh_val],
               color=["tomato", "seagreen"])
        ax.axhline(clean_val, color="steelblue", linestyle="--", linewidth=1.5,
                   label=f"Clean: {clean_val:.3g}")
        ax.legend(fontsize=8)
        ax.set_title(f"{dist_name} — {task_name}", fontsize=9)
        if task_key == "recall":
            ax.set_ylim(0, 1.1)

plt.tight_layout()
plt.show()

---
# Part 4 — Fine-Tuning YOLO on Distorted Images

### 4a. Build Distorted Training Set (GaussNoise)
We use the clean YOLO predictions as **pseudo-labels** and apply GaussNoise to create the training set.

In [ ]:
work = Path("ft_workspace")
(work / "images" / "train").mkdir(parents=True, exist_ok=True)
(work / "labels" / "train").mkdir(parents=True, exist_ok=True)


def save_yolo_label(txt_path, boxes_xyxy, cls_ids, w, h):
    lines = []
    for (x1, y1, x2, y2), c in zip(boxes_xyxy, cls_ids):
        cx = ((x1 + x2) / 2) / w
        cy = ((y1 + y2) / 2) / h
        bw = (x2 - x1) / w
        bh = (y2 - y1) / h
        lines.append(f"{int(c)} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}")
    Path(txt_path).write_text("\n".join(lines))


gauss_aug = distortions["GaussNoise"]
pseudo_names = []

for i, img in enumerate(images):
    img_np = np.array(img.convert("RGB"))
    h, w = img_np.shape[:2]

    # Get pseudo-labels from clean pretrained YOLO
    r = yolo_model.predict(img_np, conf=0.35, iou=0.5, verbose=False)[0]
    if r.boxes is None or len(r.boxes) == 0:
        continue
    boxes = r.boxes.xyxy.cpu().numpy()
    cls   = r.boxes.cls.cpu().numpy()

    # Apply GaussNoise to the image
    dist_np = gauss_aug(image=img_np)["image"]

    img_path = work / "images" / "train" / f"im_{i}.jpg"
    lbl_path = work / "labels" / "train" / f"im_{i}.txt"
    cv2.imwrite(str(img_path), cv2.cvtColor(dist_np, cv2.COLOR_RGB2BGR))
    save_yolo_label(lbl_path, boxes, cls, w, h)
    pseudo_names.append(f"im_{i}.jpg")

print(f"Training images saved: {len(pseudo_names)}")

### 4b. Create data.yaml and Fine-Tune

In [ ]:
import yaml

data_yaml = {
    "train": str((work / "images" / "train").resolve()),
    "val":   str((work / "images" / "train").resolve()),  # same set; small project
    "nc":    80,  # keep original COCO class count to reuse pretrained weights
    "names": yolo_model.names,
}
yaml_path = work / "data.yaml"
with open(yaml_path, "w") as f:
    yaml.dump(data_yaml, f)
print(f"data.yaml written: {yaml_path}")

In [ ]:
print("Fine-tuning YOLOv8n on GaussNoise distorted images...")
ft_model = YOLO("yolov8n.pt")
_ = ft_model.train(
    data=str(yaml_path),
    epochs=5,
    batch=2,
    imgsz=640,
    device=DEVICE,
    verbose=False,
    project=str(work / "runs"),
    name="finetune",
)

# Load best weights
best_pt = work / "runs" / "finetune" / "weights" / "best.pt"
if not best_pt.exists():
    best_pt = work / "runs" / "finetune" / "weights" / "last.pt"
ft_model = YOLO(str(best_pt))
print(f"Fine-tuned model loaded from: {best_pt}")

### 4c. Evaluate Fine-Tuned Model on All Distortions

In [ ]:
def yolo_recall_model(model, img_pil, gt, conf=0.25):
    img = np.array(img_pil.convert("RGB"))
    r = model.predict(img, conf=conf, verbose=False)[0]
    pred = r.boxes.xyxy.cpu().numpy() if r.boxes is not None else np.zeros((0, 4))
    return detection_recall(gt, pred)


ft_results = {}
for dist_name, aug in distortions.items():
    recalls = []
    for img, gt in zip(images, gt_boxes):
        dist_np  = apply_distortion(img, aug)
        dist_pil = Image.fromarray(dist_np)
        recalls.append(yolo_recall_model(ft_model, dist_pil, gt))
    ft_results[dist_name] = float(np.mean(recalls))
    print(f"Fine-tuned recall on {dist_name}: {ft_results[dist_name]:.3f}")

### 4d. Final Comparison Table & Charts

In [ ]:
import pandas as pd

rows = []
# Clean baseline
rows.append({"Model": "Pretrained (clean)", **{d: clean_recall_mean for d in dist_names}})

# Pretrained on distorted
rows.append({"Model": "Pretrained (distorted)",
             **{d: float(np.mean(dist_results[d]["recall"])) for d in dist_names}})

# Pretrained on enhanced
rows.append({"Model": "Pretrained + Enhancement",
             **{d: float(np.mean(enh_results[d]["recall"])) for d in dist_names}})

# Fine-tuned on distorted
rows.append({"Model": "Fine-tuned on GaussNoise",
             **ft_results})

df = pd.DataFrame(rows).set_index("Model")
print("\n=== Final YOLO Detection Recall Comparison ===")
print(df.to_string(float_format="{:.3f}".format))

In [ ]:
# ── Grouped bar chart ────────────────────────────────────────────────────────
model_labels = ["Pretrained\n(clean)", "Pretrained\n(distorted)",
                "+ Enhancement", "Fine-tuned\n(GaussNoise)"]
x = np.arange(len(dist_names))
width = 0.2
bar_colors = ["steelblue", "tomato", "seagreen", "darkorchid"]

fig, ax = plt.subplots(figsize=(12, 6))
offsets = np.linspace(-1.5 * width, 1.5 * width, 4)

for i, (row_label, color, offset) in enumerate(
        zip(model_labels, bar_colors, offsets)):
    vals = [df.iloc[i][d] for d in dist_names]
    ax.bar(x + offset, vals, width, label=row_label, color=color, alpha=0.85)

ax.axhline(clean_recall_mean, color="black", linestyle="--", linewidth=1,
           label=f"Clean baseline ({clean_recall_mean:.2f})")
ax.set_xticks(x)
ax.set_xticklabels(dist_names, fontsize=12)
ax.set_ylabel("Mean Detection Recall", fontsize=12)
ax.set_ylim(0, 1.15)
ax.set_title("Part 4 — YOLO Detection Recall: Pretrained vs Enhancement vs Fine-Tuned",
             fontsize=13, fontweight="bold")
ax.legend(loc="upper right", fontsize=10)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

print("\nProject complete!")

---
## Summary of Results

This notebook evaluated three computer vision algorithms on aerial aircraft surveillance imagery under three types of distortion:

| Phase | What was measured |
|---|---|
| **Part 1 — Baseline** | Performance of ORB, YOLO, Canny on clean images |
| **Part 2 — Distortion** | Degradation under GaussNoise, LowLight, Rain; SNR sweep curve |
| **Part 3 — Enhancement** | Recovery after NLM denoising, CLAHE, de-raining |
| **Part 4 — Fine-tuning** | YOLOv8 fine-tuned on GaussNoise distorted data |

**Key findings** (to be filled in after running):
- Most damaging distortion: *TBD after execution*
- Best enhancement strategy: *TBD after execution*
- Fine-tuning benefit: most pronounced on GaussNoise (the training distortion)